# Lab 10: Does a Crowd of Smart People Find the Truth?

### Simulations from the philosophy of science, with sliders to play with

Here is a claim that sounds obviously true: if every scientist in a field is careful and rational, the field as a whole will move toward the truth. And here is the uncomfortable thing this lab will show you. It is false. A community of perfectly reasonable people can lock onto the wrong answer, and a community of careful copycats can get stuck halfway up the mountain while a messier community climbs past them.

This is the territory of social epistemology, the part of philosophy that asks how knowledge works at the level of groups rather than individuals. We will poke at it with two small simulations. You will not write any code. Run the cells, drag the sliders, and notice when your intuitions turn out to be wrong.

### How this notebook works

You will see three kinds of cell over and over.

- **Think first.** A question that asks what you expect before you see the result. Guess out loud. The surprising cells only land if you committed to a prediction.
- **Play.** A cell with sliders. Most of these have a "Run interact" button. Set the sliders, press the button, watch what changes.
- **What just happened.** A plain-language read of the result, then a philosophical thread worth pulling.

Run the cells in order, top to bottom. The simulations are random, so your numbers will wobble a little from run to run. That wobble is part of the lesson.

## Part 0: Setup

Press play on the next cell and wait for "Ready". Nothing to read.

In [ ]:
!pip install numpy matplotlib networkx ipywidgets --quiet

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import product
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from ipywidgets import interact, interact_manual

np.random.seed(0)
print('Ready. You can keep going.')

## Part 1: When talking more makes a group dumber

In the early 2000s a lot of depression research piled onto one family of drugs, the SSRIs, because the first trials looked good. Many labs quietly dropped the alternatives. Years later, ketamine turned out to help people that nothing else could reach, working through a mechanism the field had nearly walked away from.

Ask two questions about that. Was it reasonable for each lab to chase the approach that looked best at the time? Pretty much, yes. Did the field as a whole make a good call? No. That gap between sensible individuals and a community that misses the truth is what the philosopher Kevin Zollman put into a model in 2007, and it is what we are going to rebuild.

### Think first

Picture a community of scientists choosing between two methods, A and B. B is genuinely a little better, but nobody knows that yet. They run experiments, get noisy results, and tell their colleagues what they found.

Now the key question. Suppose you could choose how connected this community is. In one version everyone hears everyone else's results immediately. In another, people only talk to a couple of neighbours. Which community do you bet finds the better method more often?

Most people say the fully connected one. Hold that thought.

### Meet the simulated scientist

You do not need to read the code in the next cell. Here is all it does in words.

Each scientist keeps a running tally of wins and losses for each method, and prefers whichever method has done better so far. When a colleague shares results, the scientist just adds those wins and losses to their own tally. That is it. This counting rule is actually the mathematically correct way to update a belief from evidence, so every scientist here is being perfectly rational. Keep that in mind, because the trouble that follows is not caused by anyone being foolish.

In [ ]:
class Scientist:
    'A simulated scientist who experiments, shares results, and updates beliefs. No need to read this.'

    def __init__(self, n_methods=2):
        # Start with no opinion: a 1-1 tally for each method
        self.hits   = [1] * n_methods
        self.misses = [1] * n_methods

    @property
    def beliefs(self):
        # How good each method looks right now, based on the tally
        return [(1 + h) / (2 + h + m) for h, m in zip(self.hits, self.misses)]

    def best_method(self):
        return int(np.argmax(self.beliefs))

    def run_experiment(self, true_probs):
        m       = self.best_method()
        success = int(np.random.random() < true_probs[m])
        if success:
            self.hits[m]   += 1
        else:
            self.misses[m] += 1
        return m, success, 1 - success

    def receive_evidence(self, method, successes, failures):
        # Hear a colleague's results: just add them to our own tally
        self.hits[method]   += successes
        self.misses[method] += failures

print('Our simulated scientist is ready.')

### Play: watch one mind change

Drag the sliders to decide what a colleague reports about method B, then watch the scientist's beliefs move. This one updates live, no button needed.

Try giving B a few wins and no losses. Notice how little it takes to flip someone's preference. Now imagine that flipped opinion spreading through a whole network.

In [ ]:
def belief_demo(B_wins=8, B_losses=1):
    s = Scientist()
    before = s.beliefs
    s.receive_evidence(method=1, successes=B_wins, failures=B_losses)
    after = s.beliefs
    pref = 'B' if s.best_method() == 1 else 'A'
    print('Before hearing anything:')
    print('   belief in A = {:.2f}      belief in B = {:.2f}'.format(before[0], before[1]))
    print()
    print('A colleague reports on method B: {} wins, {} losses'.format(B_wins, B_losses))
    print()
    print('After updating:')
    print('   belief in A = {:.2f}      belief in B = {:.2f}'.format(after[0], after[1]))
    print('   now prefers method:', pref)

interact(belief_demo,
         B_wins=widgets.IntSlider(min=0, max=20, value=8, description='B wins'),
         B_losses=widgets.IntSlider(min=0, max=20, value=1, description='B losses'))

### Three ways to wire a community together

We will test three shapes of communication network.

| Network | Who talks to whom | How fast news travels |
|---------|-------------------|------------------------|
| Complete | everyone hears everyone | instant, the whole field at once |
| Cycle | a ring, each person has two neighbours | slow, news creeps around the loop |
| Wheel | one hub connected to all, plus a ring | fast through the hub, slow on the rim |

The next cell just draws them. Run it.

In [ ]:
def make_network(topology, n):
    if topology == 'complete': return nx.complete_graph(n)
    if topology == 'cycle':    return nx.cycle_graph(n)
    if topology == 'wheel':    return nx.wheel_graph(n)
    raise ValueError('Unknown topology: ' + topology)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
descs = {'complete': 'Complete\n(everyone talks to everyone)',
         'cycle':    'Cycle\n(a ring of neighbours)',
         'wheel':    'Wheel\n(hub plus a ring)'}
for ax, topo in zip(axes, ['complete', 'cycle', 'wheel']):
    G   = make_network(topo, 10)
    pos = nx.circular_layout(G)
    nx.draw(G, pos, ax=ax, node_size=350, node_color='steelblue',
            edge_color='#aaaaaa', with_labels=False)
    ax.set_title(descs[topo], fontsize=10)
plt.suptitle('Three communication networks', fontsize=13)
plt.tight_layout()
plt.show()

### The engine

The next cell runs the actual simulation: every round each scientist experiments, then shares results with their neighbours only, then everyone updates. Run it and move on. No need to read it.

In [ ]:
def run_simulation(n, topology, p_A, p_B, n_rounds):
    G          = make_network(topology, n)
    scientists = [Scientist() for _ in range(n)]
    true_probs = [p_A, p_B]
    history    = []
    for _ in range(n_rounds):
        results = [s.run_experiment(true_probs) for s in scientists]
        for i, s in enumerate(scientists):
            for j in G.neighbors(i):
                method, succ, fail = results[j]
                s.receive_evidence(method, succ, fail)
        n_on_B = sum(1 for s in scientists if s.best_method() == 1)
        history.append(n_on_B / n)
    final = [s.best_method() for s in scientists]
    if   all(m == 1 for m in final): converged = True
    elif all(m == 0 for m in final): converged = False
    else:                            converged = None
    return history, converged

def run_many(n, topology, p_A, p_B, n_sims, n_rounds):
    n_correct = n_wrong = 0
    histories = []
    for _ in range(n_sims):
        hist, conv = run_simulation(n, topology, p_A, p_B, n_rounds)
        histories.append(hist)
        if conv is True:  n_correct += 1
        if conv is False: n_wrong   += 1
    return {'p_correct': n_correct / n_sims,
            'p_wrong':   n_wrong   / n_sims,
            'p_none':    (n_sims - n_correct - n_wrong) / n_sims,
            'mean_traj': np.mean(histories, axis=0)}

print('Engine ready.')

### Play: the main experiment

Set the dials and press Run interact. You are choosing how many scientists there are, how much better method B truly is, how many rounds they get, and how many times we repeat the whole thing to average out luck.

Start with the defaults. Then make B only barely better (drag "B is better by" down toward 0.001) and run again. Watch what happens to the fully connected community.

In [ ]:
def zollman_experiment(scientists=8, B_is_better_by=0.01, rounds=200, runs=40):
    p_A, p_B = 0.5, 0.5 + B_is_better_by
    topos = ['complete', 'cycle', 'wheel']
    res = {t: run_many(scientists, t, p_A, p_B, runs, rounds) for t in topos}

    x = np.arange(3); w = 0.35
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.bar(x - w/2, [res[t]['p_correct'] for t in topos], w,
            label='found the better method', color='steelblue')
    ax1.bar(x + w/2, [res[t]['p_wrong'] for t in topos], w,
            label='locked onto the worse one', color='tomato')
    ax1.set_xticks(x); ax1.set_xticklabels(['Complete', 'Cycle', 'Wheel'])
    ax1.set_ylabel('fraction of runs'); ax1.set_ylim(0, 1); ax1.legend(fontsize=9)
    ax1.set_title('Which network finds the truth?')

    colors = {'complete': 'tomato', 'cycle': 'steelblue', 'wheel': 'seagreen'}
    for t in topos:
        ax2.plot(res[t]['mean_traj'], color=colors[t], label=t.capitalize(), linewidth=2)
    ax2.set_xlabel('round'); ax2.set_ylabel('average fraction using method B')
    ax2.set_ylim(-0.02, 1.05); ax2.legend(fontsize=9)
    ax2.set_title('How the better method spreads over time')
    plt.tight_layout(); plt.show()

    for t in topos:
        print('{:9s} found the better method in {:.0%} of runs'.format(t, res[t]['p_correct']))

interact_manual(zollman_experiment,
    scientists=widgets.IntSlider(min=4, max=12, value=8, description='scientists'),
    B_is_better_by=widgets.FloatSlider(min=0.001, max=0.2, step=0.005, value=0.01,
                                       readout_format='.3f', description='B better by'),
    rounds=widgets.IntSlider(min=50, max=400, step=50, value=200, description='rounds'),
    runs=widgets.IntSlider(min=20, max=100, step=10, value=40, description='runs'))

### What just happened

When the evidence is faint, the fully connected community often does worst. Here is why. Everyone sees the same early results at the same time. If method A gets lucky in the first few rounds, that luck reaches every single scientist at once, they all swing to A together, and then nobody is running B anymore to discover that it was actually better. The community agrees its way into the wrong answer.

The ring does better, and the reason is almost paradoxical. Because news travels slowly, different pockets of the community hold different opinions for a while. Some pockets keep testing B even after others gave up on it. That stubborn local disagreement is exactly what gives the better method enough chances to prove itself.

Zollman's name for the useful ingredient is transient diversity: a community reasons better when it does not converge too fast. The thought is old. John Stuart Mill argued in 1859 that even a wrong opinion is worth keeping in play because it keeps the truth from hardening into dead dogma. What the simulation adds is that this is not only a political nicety. It is a structural property of groups that find things out, and you can watch it switch on and off with a slider.

### Think first

So is connectivity just bad? That cannot be the whole story, or science would forbid conferences. Predict this: as method B becomes clearly, obviously better, what happens to the gap between the fast network and the slow one?

In [ ]:
def evidence_strength_sweep(scientists=8, rounds=200, runs=40):
    deltas = [0.001, 0.005, 0.02, 0.05, 0.1]
    out = {'complete': [], 'cycle': []}
    for d in deltas:
        for t in ['complete', 'cycle']:
            out[t].append(run_many(scientists, t, 0.5, 0.5 + d, runs, rounds)['p_correct'])
    xs = range(len(deltas))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(xs, out['complete'], color='tomato', marker='o', linewidth=2,
            label='Complete (everyone shares)')
    ax.plot(xs, out['cycle'], color='steelblue', marker='o', linewidth=2,
            label='Cycle (share with two neighbours)')
    ax.set_xticks(list(xs)); ax.set_xticklabels([str(d) for d in deltas])
    ax.set_xlabel('how much better method B really is')
    ax.set_ylabel('fraction of runs that found B')
    ax.set_ylim(0, 1.05); ax.legend(fontsize=9)
    ax.set_title('The gap between the networks closes as the evidence gets clearer')
    plt.tight_layout(); plt.show()

interact_manual(evidence_strength_sweep,
    scientists=widgets.IntSlider(min=4, max=12, value=8, description='scientists'),
    rounds=widgets.IntSlider(min=50, max=400, step=50, value=200, description='rounds'),
    runs=widgets.IntSlider(min=20, max=80, step=10, value=40, description='runs'))

### What Part 1 leaves you with

The fast network only hurts when the signal is weak. When B is clearly better, everyone finds it no matter how they are wired. So the lesson is not "talk less". It is that how a community is connected matters most exactly when the evidence is hardest to read, which is to say, at the research frontier where it matters most.

This is why real institutions deliberately slow some information down. Preregistration, blind review, independent replication before a result is trusted: each one keeps the field from stampeding on the first promising hint. They are transient diversity, built into the rules.

## Part 2: Why a field needs its mavericks

Part 1 was about how scientists talk to each other. Part 2 is about how they decide what to try next, and it comes from work by Jingyi Wu and others on the division of cognitive labor.

Here is the puzzle. Suppose one lab is clearly ahead of everyone else. Should the rest of the field drop what they are doing and copy it? It feels efficient. But we are going to see that a community where everyone chases the current leader can march straight into a dead end, while a community that keeps some people wandering off on their own does better. On hard problems, anyway.

### Think first

A research problem is hard. One approach is clearly out in front right now. Which community do you expect to end up closest to the best possible answer:

one where everyone copies the current leader, one where each person copies any colleague who happens to be ahead of them, or a mix of the two?

Make a call before you run anything.

### The landscape picture

Think of every possible theory as a spot on a hilly landscape, where height means how good the theory is. Solving the problem means reaching the highest peak.

A smooth landscape has one hill. Wander uphill from anywhere and you reach the top. An easy problem.

A rugged landscape has many separate hills of different heights. Wander uphill and you reach a peak, but probably not the tallest one, and from there every direction is down, so you are stuck. A hard problem.

One dial, called K, sets the ruggedness. K is how tangled the assumptions in a theory are, how much each one's value depends on the others. Low K is smooth. High K is a mess of false summits. Run the next cell to build the landscape machinery, then we will play with K.

In [ ]:
class NKLandscape:
    'A hilly landscape over theories. No need to read this.'

    def __init__(self, N, K, seed=None):
        assert 0 <= K < N, 'K must be between 0 and N-1'
        self.N, self.K = N, K
        rng = np.random.RandomState(seed)
        self.interactions = []
        for i in range(N):
            others = [j for j in range(N) if j != i]
            deps   = sorted(rng.choice(others, K, replace=False).tolist())
            self.interactions.append([i] + deps)
        self.tables = [rng.random(2 ** (K + 1)) for _ in range(N)]

    def fitness(self, theory):
        total = 0.0
        for i in range(self.N):
            bits  = tuple(theory[j] for j in self.interactions[i])
            idx   = int(''.join(map(str, bits)), 2)
            total += self.tables[i][idx]
        return total / self.N

    def neighbors(self, theory):
        result = []
        for i in range(self.N):
            nb    = list(theory)
            nb[i] = 1 - nb[i]
            result.append(nb)
        return result

    def is_local_max(self, theory):
        f = self.fitness(theory)
        return all(self.fitness(nb) <= f for nb in self.neighbors(theory))

    def count_local_maxima(self):
        return sum(1 for bits in product([0, 1], repeat=self.N)
                   if self.is_local_max(list(bits)))

    def global_max(self):
        best_f, best_t = -1.0, None
        for bits in product([0, 1], repeat=self.N):
            f = self.fitness(list(bits))
            if f > best_f:
                best_f, best_t = f, list(bits)
        return best_t, best_f

print('Landscape ready.')

### Play: feel the ruggedness

Drag K and press Run interact. The number you get back is how many separate peaks the landscape tends to have. One peak means a lone researcher can always climb to the top. Many peaks means a lone researcher almost always gets stranded on a lesser one.

In [ ]:
def ruggedness_demo(K=2):
    counts = [NKLandscape(N=8, K=K, seed=s).count_local_maxima() for s in range(12)]
    avg = float(np.mean(counts))
    print('With ruggedness K = {} (each assumption tangled with {} others):'.format(K, K))
    print('   the landscape has about {:.1f} separate peaks'.format(avg))
    if K == 0:
        print('   one single peak. Climbing uphill always reaches the top.')
    elif avg < 4:
        print('   still fairly gentle. Most lone climbers reach the top.')
    else:
        print('   lots of false summits. A lone climber usually gets stuck.')

interact_manual(ruggedness_demo,
    K=widgets.IntSlider(min=0, max=6, value=2, description='ruggedness K'))

### Two ways to learn from your colleagues

Every scientist also tries small improvements on their own, flipping one assumption and keeping it if the theory got better. That is private hill-climbing. The interesting part is what they do socially.

Copy the best: if anyone is ahead of me, I jump to whoever is highest. Fast. Everyone piles onto the leader.

Copy any improvement: from everyone who is ahead of me, I pick one at random and jump to them. Slower, and the randomness is the whole point. Different people end up chasing different colleagues, so the community stays spread out across the landscape instead of collapsing onto one spot.

The next cell builds these scientists and the community that runs them. Run it, then we compete the strategies.

In [ ]:
class ScientistNK:
    'A scientist exploring the landscape by tinkering and by copying peers.'

    def __init__(self, landscape):
        self.landscape = landscape
        self.theory    = list(np.random.randint(0, 2, landscape.N))
        self.fit       = landscape.fitness(self.theory)

    def explore_locally(self):
        i        = np.random.randint(0, self.landscape.N)
        trial    = list(self.theory)
        trial[i] = 1 - trial[i]
        f_trial  = self.landscape.fitness(trial)
        if f_trial > self.fit:
            self.theory = trial
            self.fit    = f_trial

    def update_best(self, peer_theories, peer_fitnesses):
        if not peer_fitnesses: return
        best = int(np.argmax(peer_fitnesses))
        if peer_fitnesses[best] > self.fit:
            self.theory = list(peer_theories[best])
            self.fit    = peer_fitnesses[best]

    def update_better(self, peer_theories, peer_fitnesses):
        better = [i for i, f in enumerate(peer_fitnesses) if f > self.fit]
        if better:
            chosen      = np.random.choice(better)
            self.theory = list(peer_theories[chosen])
            self.fit    = peer_fitnesses[chosen]

def run_community(landscape, n_agents, n_rounds, strategy, mix_ratio=0.5):
    agents = [ScientistNK(landscape) for _ in range(n_agents)]
    if strategy == 'mixed':
        strats = ['better' if np.random.random() < mix_ratio else 'best' for _ in range(n_agents)]
    else:
        strats = [strategy] * n_agents
    history = [np.mean([a.fit for a in agents])]
    for _ in range(n_rounds):
        for a in agents:
            a.explore_locally()
        theories  = [a.theory for a in agents]
        fitnesses = [a.fit    for a in agents]
        for i, (agent, strat) in enumerate(zip(agents, strats)):
            pts = theories[:i]  + theories[i+1:]
            pfs = fitnesses[:i] + fitnesses[i+1:]
            if strat == 'best':
                agent.update_best(pts, pfs)
            else:
                agent.update_better(pts, pfs)
        history.append(np.mean([a.fit for a in agents]))
    return history, history[-1]

print('Community ready.')

### Play: the strategy showdown

Set the ruggedness, the share of "explorers" who copy any improvement rather than the leader, the number of scientists, and the number of rounds. Press Run interact.

Try K = 1 first, a gentle landscape. Then crank K up to 5 or 6 and run again. Watch which strategy pulls ahead as the problem gets harder. Then play with the percentage of explorers and see if you can beat the default mix.

In [ ]:
def strategy_showdown(ruggedness_K=4, percent_explorers=50, scientists=10, rounds=60):
    trials = 6
    mix = percent_explorers / 100.0
    trajs = {'best': [], 'better': [], 'mixed': []}
    for seed in range(trials):
        land = NKLandscape(N=8, K=ruggedness_K, seed=seed)
        _, gmax = land.global_max()
        for strat in ['best', 'better', 'mixed']:
            hist, _ = run_community(land, scientists, rounds, strat, mix_ratio=mix)
            trajs[strat].append([h / gmax for h in hist])
    colors = {'best': 'tomato', 'better': 'steelblue', 'mixed': 'seagreen'}
    labels = {'best': 'Best (everyone copies the leader)',
              'better': 'Better (copy any colleague ahead of you)',
              'mixed': 'Mixed community'}
    fig, ax = plt.subplots(figsize=(10, 5))
    for strat in ['best', 'better', 'mixed']:
        ax.plot(np.mean(trajs[strat], axis=0), color=colors[strat],
                label=labels[strat], linewidth=2)
    ax.set_xlabel('round'); ax.set_ylabel('how close to the best possible theory')
    ax.set_ylim(0.3, 1.05); ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)
    ax.legend(fontsize=9)
    ax.set_title('Strategies on a landscape with ruggedness K = {}'.format(ruggedness_K))
    plt.tight_layout(); plt.show()
    print('Final score (1.00 = found the best possible theory):')
    for strat in ['best', 'better', 'mixed']:
        print('   {:7s} {:.2f}'.format(strat, float(np.mean([t[-1] for t in trajs[strat]]))))

interact_manual(strategy_showdown,
    ruggedness_K=widgets.IntSlider(min=0, max=6, value=4, description='ruggedness K'),
    percent_explorers=widgets.IntSlider(min=0, max=100, step=10, value=50, description='% explorers'),
    scientists=widgets.IntSlider(min=4, max=20, value=10, description='scientists'),
    rounds=widgets.IntSlider(min=20, max=120, step=20, value=60, description='rounds'))

### What just happened

On the gentle landscape the three strategies basically tie. One hill, everyone climbs it, the social rule barely matters.

On the rugged landscape the mixed community usually wins, and the all-copy-the-leader community usually does worst. The reason is the one we set up earlier. When everyone copies the single leader, the whole community piles onto one peak. If it is not the tallest one, they are all stuck together with nowhere up to go. When people copy different colleagues at random, the community stays scattered across several peaks, so somebody is usually parked on higher ground, and the others can climb toward them.

This is the division of cognitive labor. A field does better when it does not put all its effort behind the current favourite, when some researchers keep working approaches that look worse right now. We tend to frame that as a matter of fairness or open-mindedness. The model says it is also just good design. The mavericks are not being tolerated. They are doing structural work that the followers cannot do, and the field would be worse off without them.

## Part 3: The two halves together

Both simulations push on the same comfortable assumption, that a group of rational people adds up to a rational group.

In Part 1 every scientist updated on evidence perfectly, and the community still talked itself into the wrong answer when it was wired too tightly. The failure lived in the structure of communication, not in any person.

In Part 2 copying the leader was a perfectly sensible move for any individual, and a community of such individuals got stranded on a false summit. The fix was not smarter scientists. It was a less uniform community.

Put together: epistemic diversity, whether in who-talks-to-whom or in who-tries-what, is not a decoration on good collective reasoning. It is part of the machinery. That is a genuinely philosophical conclusion, and it has teeth. It means a field can be full of brilliant, honest people and still fail, and that some of the things we do in the name of efficiency and consensus are quietly making the group dumber.

### Reflection

Write a few sentences on each. No answer keys. These are the questions the lab was built to raise.

1. A company puts every internal experiment on a shared live dashboard so the whole staff sees results the moment they land. Using Part 1, when would you expect this to help the company find good ideas, and when would you expect it to backfire?

2. The explorer strategy works well, so why does so little real science run that way? What is it about how careers, grants, and publications work that pushes everyone toward copying the leader?

3. Both models assumed people reason without bias. Now add a thumb on the scale, say everyone overweights results from famous labs. Do you think diversity helps more under that bias, or less? Argue it.

4. Language models are trained on the published record. If a field converged too early on a wrong answer, that record is skewed. Does a model trained on it inherit the mistake, and does AI summarising the literature make the Zollman trap better or worse?

5. Mill defended keeping wrong opinions alive so the truth stays vivid. Zollman gives a different defence of the same diversity, one about reaching the truth at all rather than understanding it well. Are these the same argument wearing two coats, or two different reasons that happen to agree? Which moves you more?

**Your answers:**

1.

2.

3.

4.

5.

## Wrap-up

What you did:

- Ran a community of perfectly rational scientists and watched the most connected version talk itself into the wrong answer, then watched that failure vanish as the evidence got clearer.
- Ran scientists across rugged landscapes and watched a mixed community of leader-followers and lone explorers beat a community that all chased the leader.
- Arrived at one idea from two directions: diversity in a community is structural, not cosmetic, and individual rationality does not guarantee a rational group.

The models are toys. The institutions they echo are not: peer review, preregistration, replication, and the slightly inefficient habit of funding people who disagree with the current consensus. If this lab makes one of those look different to you than it did this morning, it did its job.

---

**Author:** [Aniket Ghosh](https://www.linkedin.com/in/aniketghosh-/)